In [10]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

In [11]:
# 将建筑和楼层进行合并，形成新的编码


# 1. 读取训练数据（路径请按需修改）
train_df = pd.read_csv('../../UJIIndoorLoc/trainingData.csv')
valid_df = pd.read_csv('../../UJIIndoorLoc/validationData.csv')
train_df_noisy1 = pd.read_csv('../../data/train_noisy1.csv')

# train_df = pd.concat([train_df, train_df_noisy1], ignore_index=True)
# total_df = pd.concat([train_df, valid_df], ignore_index=True)

# 2. 创建联合标签列（如 "2_3" 表示 BUILDINGID=2 且 FLOOR=3）
train_df['location_label'] = train_df['BUILDINGID'].astype(str) + '_' + train_df['FLOOR'].astype(str)
valid_df['location_label'] = valid_df['BUILDINGID'].astype(str) + '_' + valid_df['FLOOR'].astype(str)
# total_df['location_label'] = total_df['BUILDINGID'].astype(str) + '_' + total_df['FLOOR'].astype(str)

# 3. 将联合标签进行整数编码
label_encoder = LabelEncoder()
train_df['location_label_encoded'] = label_encoder.fit_transform(train_df['location_label'])
valid_df['location_label_encoded'] = label_encoder.transform(valid_df['location_label'])  # 改这里
# total_df['location_label_encoded'] = label_encoder.fit_transform(valid_df['location_label'])
total_df = pd.concat([train_df, valid_df])
# 4. 保存为 CSV 文件
# train_df.to_csv('./data/processed_train.csv', index=False)
# valid_df.to_csv('./data/processed_valid.csv', index=False)

# print("✅ 文件已成功保存为 'processed_train.csv'")
# print("✅ 文件已成功保存为 'processed_validat.csv'")

In [12]:
# 从 total_df 中提取特征和标签
X = total_df[total_df.columns[:520]].to_numpy()  # 特征：520个WAP
y_floor = total_df['location_label_encoded'].to_numpy()  # 楼层标签
y_longitude = total_df['LONGITUDE'].to_numpy()  # 经度标签
y_latitude = total_df['LATITUDE'].to_numpy()  # 纬度标签
z = total_df['SPACEID'].to_numpy()  # 用于分层的 SPACEID

# 第一步：先切 20% 测试集（通过 SPACEID 分层）
X_temp, X_test, y_floor_temp, y_floor_test, y_lon_temp, y_lon_test, y_lat_temp, y_lat_test, z_temp, z_test = train_test_split(
    X, y_floor, y_longitude, y_latitude, z,
    test_size=0.2,               # 20% → 测试集
    random_state=2812,
    stratify=z
)

# 第二步：在剩下的 80% 里切 10% 验证集 (0.8 * 1/8 = 0.1)
X_train, X_valid, y_floor_train, y_floor_valid, y_lon_train, y_lon_valid, y_lat_train, y_lat_valid = train_test_split(
    X_temp, y_floor_temp, y_lon_temp, y_lat_temp,
    test_size=1/8,               # 10% → 验证集
    random_state=2812,
    stratify=z_temp
)

# 为了兼容后续代码，创建别名
training_data = X_train
valid_data = X_valid
test_data = X_test

training_floors = y_floor_train
valid_floors = y_floor_valid
test_floors = y_floor_test

training_longitude = y_lon_train
valid_longitude = y_lon_valid
test_longitude = y_lon_test

training_latitude = y_lat_train
valid_latitude = y_lat_valid
test_latitude = y_lat_test

print(f"数据集划分完成（按 SPACEID 分层）：")
print(f"  训练集: {X_train.shape[0]} 条 (70%)")
print(f"  验证集: {X_valid.shape[0]} 条 (10%)")
print(f"  测试集: {X_test.shape[0]} 条 (20%)")



数据集划分完成（按 SPACEID 分层）：
  训练集: 14733 条 (70%)
  验证集: 2105 条 (10%)
  测试集: 4210 条 (20%)


In [13]:
# 对数据集进行填充
# 将 100（表示没有检测到信号）替换为 -105
# 理由：真实 RSSI 信号范围为 -104 到 0 dBm
# 用 -105 表示"无信号"，比最弱的真实信号 -104 还低
# 这样归一化后，无信号变为 0（最小值），符合"信号最弱"的语义

NO_SIGNAL_VALUE = 100  # 原始数据中表示无信号的值
REPLACE_VALUE = -105   # 替换后的值（比最弱信号 -104 更低）

# 替换训练集、验证集、总数据集中的无信号值
training_data = np.where(training_data == NO_SIGNAL_VALUE, REPLACE_VALUE, training_data)
valid_data = np.where(valid_data == NO_SIGNAL_VALUE, REPLACE_VALUE, valid_data)
test_data = np.where(test_data == NO_SIGNAL_VALUE, REPLACE_VALUE, test_data)
total_data = np.where(X == NO_SIGNAL_VALUE, REPLACE_VALUE, X)

print(f"替换完成！无信号值 {NO_SIGNAL_VALUE} 已替换为 {REPLACE_VALUE}")
print(f"当前数据范围: min={total_data.min()}, max={total_data.max()}")


替换完成！无信号值 100 已替换为 -105
当前数据范围: min=-105, max=0


In [14]:
# 数据归一化处理
import sys
sys.path.append('..')

from utill.data_standar import normalize_rssi, normalize_coords, normalize_test_or_valid_data
# 数据标准化,从总数居获取最大值最小值
X_totalCo_cnn, X_min, X_max = normalize_rssi(total_data)
print(X_min, X_max)
# 训练集特征标准化
training_data = normalize_test_or_valid_data(X_min, X_max, training_data)
# 验证集特征标准化
valid_data = normalize_test_or_valid_data(X_min, X_max, valid_data)
# 测试集特征标准化
test_data = normalize_test_or_valid_data(X_min, X_max, test_data)
# 全集
total_data = normalize_test_or_valid_data(X_min, X_max, total_data)

-105 0


# PCA方式降维

In [15]:
# 使用pca进行降维
from sklearn.decomposition import PCA
pca = PCA(n_components=30)
X_trainF_reduced = pca.fit_transform(training_data)
X_validF_reduced = pca.transform(valid_data) 
X_testF_reduced = pca.transform(test_data)

In [16]:
print(f"X_testF_reduced 长度: {len(X_testF_reduced)}")
print(f"test_floors 长度: {len(test_floors)}")

X_testF_reduced 长度: 4210
test_floors 长度: 4210


In [17]:
from KNN import KNN_clf

In [18]:
# Testing Floor Classification (pca n_components=10)
for n in range(1,6):
    knn = KNN_clf(n_neighbours=n)
    knn.fit(X_trainF_reduced, training_floors)
    preds = knn.predict(X_testF_reduced)
    print(f'Model Accuracy (KNN={n}): {knn.accuracy_metric(preds, test_floors) *100}%' )

Model Accuracy (KNN=1): 98.71733966745843%
Model Accuracy (KNN=2): 98.71733966745843%
Model Accuracy (KNN=3): 98.69358669833728%
Model Accuracy (KNN=4): 98.69358669833728%
Model Accuracy (KNN=5): 98.38479809976248%


In [19]:
from KNN import KNN_reg

In [20]:
# Longitude Regression (pca n_components=10)
for n in range(1,6):
    knn = KNN_reg(n_neighbours=n)
    knn.fit(X_trainF_reduced, training_longitude)
    preds = knn.predict(X_testF_reduced)
    print(f'(KNN={n})')
    print(f'Test MSE: {knn.MSE_metric(preds, test_longitude)}')
    print(f'Test R^2: {knn.r2_metric(preds, test_longitude)}\n')


(KNN=1)
Test MSE: 136.75940319730216
Test R^2: 0.9911134075706222

(KNN=2)
Test MSE: 3365.483039262905
Test R^2: 0.7813117387272777

(KNN=3)
Test MSE: 5815.212264204034
Test R^2: 0.622128935384824

(KNN=4)
Test MSE: 7341.914006942075
Test R^2: 0.5229242311250337

(KNN=5)
Test MSE: 8336.843718926048
Test R^2: 0.45827394281159284



In [21]:
#  Latitude Regression (pca n_components=10)
for n in range(1,6):
    knn = KNN_reg(n_neighbours=n)
    knn.fit(X_trainF_reduced, training_latitude)
    preds = knn.predict(X_testF_reduced)
    print(f'(KNN={n})')
    print(f'Test MSE: {knn.MSE_metric(preds, test_latitude)}')

(KNN=1)
Test MSE: 48.74648084806528
(KNN=2)
Test MSE: 1405387472.083252
(KNN=3)
Test MSE: 2498466570.002682
(KNN=4)
Test MSE: 3162121743.5727444
(KNN=5)
Test MSE: 3597791843.338863
